# 01 · Carga del corpus

Este notebook corresponde al **apartado 4.1.1 (Corpus utilizado)** de la memoria y aporta además la comprobación de duraciones citada en el **apartado 4.1.3**.

| | |
|---|---|
| **Entrada** | `data/groove/info.csv` y los 1.150 archivos MIDI del Groove MIDI Dataset |
| **Salida** | `results/01_info_loaded.csv`, `results/01_load_summary.csv` |

## 0. Configuración

Localiza la carpeta raíz del proyecto (la que contiene `data/`), define las rutas y muestra las versiones de Python y de las bibliotecas empleadas.

In [60]:
from pathlib import Path
import platform
import pandas as pd
import pretty_midi

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "groove"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| pretty_midi:", pretty_midi.__version__)

Python: 3.13.18
pandas: 3.0.6 | pretty_midi: 0.2.11.post0


## 1. Lectura del archivo de metadatos

Se lee `info.csv` y se comprueba que contiene los once campos descritos en la documentación oficial del corpus.

In [61]:
info = pd.read_csv(DATA_DIR / "info.csv")

EXPECTED_COLUMNS = ["drummer", "session", "id", "style", "bpm", "beat_type",
                    "time_signature", "midi_filename", "audio_filename",
                    "duration", "split"]
missing_columns = set(EXPECTED_COLUMNS) - set(info.columns)

print("Número de registros:", len(info))
print("Campos que faltan:", missing_columns or "ninguno")
info.head(3)

Número de registros: 1150
Campos que faltan: ninguno


,drummer,session,id,style,bpm,beat_type,time_signature,midi_filename,audio_filename,duration,split
0,drummer1,drummer1/eval_session,drummer1/eval_session/1,funk/groove1,138,beat,4-4,drummer1/eval_session/1_funk-groove1_138_beat_...,drummer1/eval_session/1_funk-groove1_138_beat_...,27.872308,test
1,drummer1,drummer1/eval_session,drummer1/eval_session/10,soul/groove10,102,beat,4-4,drummer1/eval_session/10_soul-groove10_102_bea...,drummer1/eval_session/10_soul-groove10_102_bea...,37.691158,test
2,drummer1,drummer1/eval_session,drummer1/eval_session/2,funk/groove2,105,beat,4-4,drummer1/eval_session/2_funk-groove2_105_beat_...,drummer1/eval_session/2_funk-groove2_105_beat_...,36.351218,test


## 2. Comparación con la documentación oficial (Tabla 3)

Se cuentan los fragmentos de tipo *beat* y *fill* de cada partición y se comparan con los publicados en la documentación oficial, que son los que recoge la **Tabla 3** de la memoria.

In [62]:
EXPECTED = pd.DataFrame(
    {"beat": [378, 48, 77], "fill": [519, 76, 52]},
    index=["train", "validation", "test"],
)

observed = (pd.crosstab(info["split"], info["beat_type"])
              .reindex(index=EXPECTED.index, columns=EXPECTED.columns, fill_value=0))

print("Beats y fills por partición (obtenido):")
display(observed)
print("¿Coincide con la documentación oficial?:", observed.equals(EXPECTED))

Beats y fills por partición (obtenido):


,beat,fill
train,378,519
validation,48,76
test,77,52


¿Coincide con la documentación oficial?: True


## 3. Lectura de los archivos MIDI

Para cada registro se comprueba que el archivo MIDI existe, que `pretty_midi` puede abrirlo, que contiene notas y cuántas pistas tiene. Se guardan también el final de la última nota y el último evento de control, que se usan en la sección 4.

In [63]:
rows = []
for _, rec in info.iterrows():
    path = DATA_DIR / rec["midi_filename"]
    row = {"id": rec["id"], "exists": path.exists(), "loaded": False, "n_notes": 0,
           "n_tracks": 0, "n_drum_tracks": 0, "last_note_end_s": None, "last_control_s": None}
    if row["exists"]:
        try:
            pm = pretty_midi.PrettyMIDI(str(path))
            notes = [n for inst in pm.instruments for n in inst.notes]
            controls = [c for inst in pm.instruments for c in inst.control_changes]
            row.update(
                loaded=True,
                n_notes=len(notes),
                n_tracks=len(pm.instruments),
                n_drum_tracks=sum(inst.is_drum for inst in pm.instruments),
                last_note_end_s=max((n.end for n in notes), default=None),
                last_control_s=max((c.time for c in controls), default=0.0),
            )
        except Exception:
            pass
    rows.append(row)

load = pd.DataFrame(rows)

print("Archivos no encontrados:", int((~load["exists"]).sum()))
print("Archivos con error de lectura:", int((load["exists"] & ~load["loaded"]).sum()))
print("Archivos sin notas:", int((load["loaded"] & (load["n_notes"] == 0)).sum()))
print("Archivos cargados correctamente:", int((load["loaded"] & (load["n_notes"] > 0)).sum()))
print("Pistas por archivo:", load["n_tracks"].value_counts().to_dict())
print("Archivos con alguna pista no marcada como percusión:",
      int((load["n_drum_tracks"] < load["n_tracks"]).sum()))

Archivos no encontrados: 0
Archivos con error de lectura: 0
Archivos sin notas: 0
Archivos cargados correctamente: 1150
Pistas por archivo: {1: 1150}
Archivos con alguna pista no marcada como percusión: 0


## 4. Duración de los archivos (apartado 4.1.3)

La documentación indica que `duration` es la duración del MIDI en segundos. Se compara con el final de la última nota y con el último evento de control para saber qué criterio sigue. Este resultado justifica que, en el resto del trabajo, la duración se calcule a partir de las notas.

In [64]:
dur = load.merge(info[["id", "duration"]], on="id")

TOL_S = 0.01
equals_note_end = (dur["duration"] - dur["last_note_end_s"]).abs() <= TOL_S
equals_last_control = (dur["duration"] - dur["last_control_s"]).abs() <= TOL_S
controls_after_notes = dur["last_control_s"] > dur["last_note_end_s"] + TOL_S

dur["criterion"] = "otro"
dur.loc[equals_note_end, "criterion"] = "final de la última nota"
dur.loc[~equals_note_end & equals_last_control, "criterion"] = "último evento de control"

print("Valor de `duration` según el evento con el que coincide:")
print(dur["criterion"].value_counts().to_string())
print("\nArchivos con eventos de control posteriores a la última nota:")
print(" - y `duration` igual al último evento de control:",
      int((controls_after_notes & ~equals_note_end & equals_last_control).sum()))
print(" - y `duration` igual al final de la última nota:",
      int((controls_after_notes & equals_note_end).sum()))

Valor de `duration` según el evento con el que coincide:
criterion
final de la última nota     875
último evento de control    275

Archivos con eventos de control posteriores a la última nota:
 - y `duration` igual al último evento de control: 275
 - y `duration` igual al final de la última nota: 262


## 5. Resumen y guardado

Se guarda `info.csv` junto con el resultado de la carga (`01_info_loaded.csv`), que es la entrada del notebook 02, y un resumen de las comprobaciones (`01_load_summary.csv`).

In [65]:
loaded_ok = load["loaded"] & (load["n_notes"] > 0)
summary = pd.Series({
    "registros_info_csv": len(info),
    "cargados_correctamente": int(loaded_ok.sum()),
    "coincide_con_documentacion": bool(observed.equals(EXPECTED)),
    "archivos_con_una_pista_de_percusion": int(((load["n_tracks"] == 1) & (load["n_drum_tracks"] == 1)).sum()),
    "duration_igual_final_ultima_nota": int(equals_note_end.sum()),
    "duration_igual_ultimo_control": int((~equals_note_end & equals_last_control).sum()),
}, name="valor")
print(summary.to_string())

summary.to_csv(RESULTS_DIR / "01_load_summary.csv")
info.merge(load, on="id").to_csv(RESULTS_DIR / "01_info_loaded.csv", index=False)

registros_info_csv                     1150
cargados_correctamente                 1150
coincide_con_documentacion             True
archivos_con_una_pista_de_percusion    1150
duration_igual_final_ultima_nota        875
duration_igual_ultimo_control           275
